## 1. Compare four-cluster silhouette scores
This reproduces the best four-cluster candidate I tried on the current tour data. It uses distance and recorded track hours, while leaving out elapsed_span_hours. The weighted version gives recorded hours four times the weight of distance. The data comes from /Users/divyshah/Desktop/dt-cargo-vehicle-telemetry-/data/processed/tour_level/retained_tours_quality_checked.csv.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(
    root / "data/processed/tour_level/retained_tours_quality_checked.csv"
)

features = ["total_distance_km", "recorded_track_hours"]
x = df[features].apply(pd.to_numeric, errors="coerce")
x = np.log1p(x.clip(lower=0))

x = SimpleImputer(strategy="median").fit_transform(x)
x = StandardScaler().fit_transform(x)

# Use the same sample to compare each feature weighting and k.
sample_idx = np.sort(
    np.random.default_rng(42).choice(
        len(x), size=min(3000, len(x)), replace=False
    )
)

for name, weights in [
    ("equal", [1.0, 1.0]),
    ("recorded-hours-weighted", [0.5, 2.0]),
]:
    x_weighted = x * np.array(weights)
    results = []

    for k in range(2, 9):
        model = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = model.fit_predict(x_weighted)

        score = silhouette_score(
            x_weighted[sample_idx],
            labels[sample_idx],
        )
        sizes = np.bincount(labels, minlength=k)
        results.append((score, k, sizes))

    best = max(results, key=lambda row: row[0])
    print(
        name,
        "best silhouette:", round(best[0], 4),
        "k:", best[1],
        "sizes:", best[2].tolist(),
    )

equal best silhouette: 0.5989 k: 2 sizes: [6651, 6762]
recorded-hours-weighted best silhouette: 0.5977 k: 2 sizes: [5087, 8326]


## 2. Split tours at long pauses between tracks
Run this before rebuilding tour-level features. It creates a new episode whenever a track starts more than 24 hours after the prior track ended. Review the threshold before adopting it.

In [2]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

tracks = pd.read_csv(root / "data/processed/engineered_tracks.csv")

tracks["start_time"] = pd.to_datetime(
    tracks["start_time"], utc=True, errors="coerce"
)
tracks["stop_time"] = pd.to_datetime(
    tracks["stop_time"], utc=True, errors="coerce"
)

tracks = tracks.dropna(
    subset=["vehicle_id", "tour_id", "start_time", "stop_time"]
).sort_values(["vehicle_id", "tour_id", "start_time"])

tour_keys = ["vehicle_id", "tour_id"]
previous_stop = tracks.groupby(tour_keys)["stop_time"].shift()

tracks["gap_from_previous_h"] = (
    tracks["start_time"] - previous_stop
).dt.total_seconds() / 3600

MAX_GAP_HOURS = 24
tracks["episode_no"] = (
    tracks.groupby(tour_keys)["gap_from_previous_h"]
    .transform(lambda gaps: gaps.gt(MAX_GAP_HOURS).cumsum())
    .astype(int)
)

tracks["track_hours"] = (
    tracks["stop_time"] - tracks["start_time"]
).dt.total_seconds() / 3600
tracks.loc[tracks["track_hours"] < 0, "track_hours"] = np.nan

episodes = (
    tracks.groupby(tour_keys + ["episode_no"], as_index=False)
    .agg(
        start_time=("start_time", "min"),
        end_time=("stop_time", "max"),
        total_distance_km=("distance_km", "sum"),
        recorded_track_hours=("track_hours", lambda s: s.sum(min_count=1)),
        track_count=("track_id", "size"),
    )
)

episodes["elapsed_span_hours"] = (
    episodes["end_time"] - episodes["start_time"]
).dt.total_seconds() / 3600

print("Track rows:", len(tracks))
print("Trip episodes:", len(episodes))

Track rows: 151256
Trip episodes: 15289


## 3. Evaluate duration regression on a chronological holdout
This fits a global baseline using distance and vehicle ID. It screens the training examples by the 24-hour gap rule, then reports results on both the full holdout and the retrospectively eligible subset. The all-holdout result is the key check for how it handles long gaps.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(
    root / "data/processed/tour_level/retained_tours_quality_checked.csv"
)
df["start_time"] = pd.to_datetime(
    df["start_time"], utc=True, errors="coerce"
)
df["gap_h"] = df["elapsed_span_hours"] - df["recorded_track_hours"]

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(
    subset=["start_time", "vehicle_id", "total_distance_km", "elapsed_span_hours"]
).sort_values("start_time")

# Split by time before applying the training eligibility rule.
cut = int(0.80 * len(df))
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

train = train.loc[train["gap_h"] <= 24]

preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

model = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5,
        random_state=42,
    ),
)

inputs = ["vehicle_id", "total_distance_km"]
model.fit(train[inputs], train["elapsed_span_hours"])

for label, scored in [
    ("all held-out tours", test),
    ("held-out tours with observed gap <= 24h", test.loc[test["gap_h"] <= 24]),
]:
    predictions = model.predict(scored[inputs])
    actual = scored["elapsed_span_hours"]

    print(
        label,
        "n =", len(scored),
        "R² =", round(r2_score(actual, predictions), 3),
        "MAE hours =", round(mean_absolute_error(actual, predictions), 2),
        "RMSE hours =", round(mean_squared_error(actual, predictions) ** 0.5, 2),
    )

all held-out tours n = 2682 R² = -0.01 MAE hours = 57.14 RMSE hours = 340.87
held-out tours with observed gap <= 24h n = 2369 R² = 0.385 MAE hours = 2.59 RMSE hours = 4.21


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

tracks = pd.read_csv(root / "data/processed/engineered_tracks.csv")

tracks["start_time"] = pd.to_datetime(
    tracks["start_time"], utc=True, errors="coerce"
)
tracks["stop_time"] = pd.to_datetime(
    tracks["stop_time"], utc=True, errors="coerce"
)
tracks["distance_km"] = pd.to_numeric(
    tracks["distance_km"], errors="coerce"
)

keys = ["vehicle_id", "tour_id"]

tracks["track_duration_hours"] = (
    tracks["stop_time"] - tracks["start_time"]
).dt.total_seconds() / 3600

tracks["flag_missing_id_or_time"] = (
    tracks[keys].isna().any(axis=1)
    | tracks["start_time"].isna()
    | tracks["stop_time"].isna()
)
tracks["flag_invalid_duration"] = (
    tracks["track_duration_hours"].isna()
    | tracks["track_duration_hours"].le(0)
)
tracks["flag_track_over_24h"] = tracks["track_duration_hours"].gt(24)
tracks["flag_missing_distance"] = tracks["distance_km"].isna()
tracks["flag_negative_distance"] = tracks["distance_km"].lt(0)

tracks["track_speed_kmh"] = (
    tracks["distance_km"]
    / tracks["track_duration_hours"].where(
        tracks["track_duration_hours"] > 0
    )
)
tracks["flag_speed_over_120"] = tracks["track_speed_kmh"].gt(120)

# Keep unassignable rows in the review output.
flag_columns = [
    "flag_missing_id_or_time",
    "flag_invalid_duration",
    "flag_track_over_24h",
    "flag_missing_distance",
    "flag_negative_distance",
    "flag_speed_over_120",
]

tracks["track_telemetry_flagged"] = tracks[flag_columns].any(axis=1)

assignable = tracks.loc[~tracks["flag_missing_id_or_time"]].copy()
assignable = assignable.sort_values(keys + ["start_time"])

assignable["gap_from_previous_h"] = (
    assignable["start_time"]
    - assignable.groupby(keys)["stop_time"].shift()
).dt.total_seconds() / 3600

# A pause over 24 hours starts a new episode. It is a boundary, not a bad track.
MAX_GAP_HOURS = 24
assignable["new_episode"] = (
    assignable["gap_from_previous_h"] > MAX_GAP_HOURS
)
assignable["episode_no"] = (
    assignable.groupby(keys)["new_episode"].cumsum().astype(int)
)
assignable["boundary_gap_before_h"] = assignable[
    "gap_from_previous_h"
].where(assignable["new_episode"])
assignable["internal_gap_h"] = assignable[
    "gap_from_previous_h"
].where(~assignable["new_episode"])

assignable["valid_track_hours"] = assignable[
    "track_duration_hours"
].where(assignable["track_duration_hours"] > 0)

episodes = (
    assignable.groupby(keys + ["episode_no"], as_index=False)
    .agg(
        start_time=("start_time", "min"),
        end_time=("stop_time", "max"),
        total_distance_km=("distance_km", lambda s: s.sum(min_count=1)),
        recorded_track_hours=(
            "valid_track_hours",
            lambda s: s.sum(min_count=1),
        ),
        track_count=("track_id", "size"),
        max_internal_gap_h=("internal_gap_h", "max"),
        boundary_gap_before_h=("boundary_gap_before_h", "first"),
        flagged_track_count=("track_telemetry_flagged", "sum"),
    )
)

episodes["elapsed_span_hours"] = (
    episodes["end_time"] - episodes["start_time"]
).dt.total_seconds() / 3600
episodes["telemetry_flagged"] = episodes["flagged_track_count"] > 0

output_dir = root / "data/processed/tour_level"
output_dir.mkdir(parents=True, exist_ok=True)

tracks.loc[tracks["track_telemetry_flagged"]].to_csv(
    output_dir / "telemetry_flagged_tracks.csv", index=False
)
episodes.to_csv(
    output_dir / "telemetry_flagged_episodes.csv", index=False
)

print("Raw tracks:", len(tracks))
print("Unassignable tracks:", int(tracks["flag_missing_id_or_time"].sum()))
print("Flagged tracks:", int(tracks["track_telemetry_flagged"].sum()))
print("Episodes:", len(episodes))
print("Flagged episodes:", int(episodes["telemetry_flagged"].sum()))

Raw tracks: 151598
Unassignable tracks: 342
Flagged tracks: 635
Episodes: 15289
Flagged episodes: 276


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=["start_time", "vehicle_id", "total_distance_km", "elapsed_span_hours"]
).sort_values("start_time").reset_index(drop=True)

cut = int(0.80 * len(episodes))
train = episodes.iloc[:cut].copy()
test = episodes.iloc[cut:].copy()

# Use telemetry flags for training eligibility. Do not filter the main test score.
train = train.loc[~train["telemetry_flagged"]].copy()

inputs = ["vehicle_id", "total_distance_km"]

preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

model = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5,
        random_state=42,
    ),
)

model.fit(train[inputs], train["elapsed_span_hours"])

def report_score(label, scored):
    predictions = model.predict(scored[inputs])
    actual = scored["elapsed_span_hours"]

    print(
        label,
        "n =", len(scored),
        "R² =", round(r2_score(actual, predictions), 3),
        "MAE hours =", round(mean_absolute_error(actual, predictions), 2),
        "RMSE hours =", round(mean_squared_error(actual, predictions) ** 0.5, 2),
    )

scorable_test = test.dropna(subset=inputs + ["elapsed_span_hours"])
clean_test = scorable_test.loc[~scorable_test["telemetry_flagged"]]

print("Training episodes:", len(train))
print("Scorable held-out episodes:", len(scorable_test))
report_score("All held-out episodes", scorable_test)
report_score("Unflagged held-out episodes (diagnostic)", clean_test)

Training episodes: 12069
Scorable held-out episodes: 3058
All held-out episodes n = 3058 R² = 0.472 MAE hours = 13.18 RMSE hours = 23.73
Unflagged held-out episodes (diagnostic) n = 2944 R² = 0.448 MAE hours = 12.54 RMSE hours = 22.92


In [6]:
import numpy as np
import pandas as pd

audit = scorable_test.copy()
audit["prediction_h"] = model.predict(audit[inputs])
audit["absolute_error_h"] = (
    audit["elapsed_span_hours"] - audit["prediction_h"]
).abs()

# These are diagnostic fields, not model inputs.
audit["elapsed_minus_recorded_h"] = (
    audit["elapsed_span_hours"] - audit["recorded_track_hours"]
)
audit["max_internal_gap_h"] = audit["max_internal_gap_h"].fillna(0)

audit["internal_gap_band"] = pd.cut(
    audit["max_internal_gap_h"],
    bins=[-np.inf, 0, 1, 3, 6, 12, 24, np.inf],
    labels=["overlap", "0–1 h", "1–3 h", "3–6 h", "6–12 h", "12–24 h", ">24 h"],
)

gap_summary = (
    audit.groupby("internal_gap_band", observed=True)
    .agg(
        episodes=("elapsed_span_hours", "size"),
        median_internal_gap_h=("max_internal_gap_h", "median"),
        median_absolute_error_h=("absolute_error_h", "median"),
        mean_absolute_error_h=("absolute_error_h", "mean"),
        p90_absolute_error_h=("absolute_error_h", lambda s: s.quantile(0.90)),
    )
)

print(gap_summary.round(2))

print("\nSpearman correlations:")
print(
    audit[
        ["max_internal_gap_h", "elapsed_minus_recorded_h", "absolute_error_h"]
    ].corr(method="spearman").round(3)
)

                   episodes  median_internal_gap_h  median_absolute_error_h  \
internal_gap_band                                                             
overlap                 732                   0.00                     3.05   
0–1 h                   808                   0.54                     2.96   
1–3 h                   474                   1.62                     2.20   
3–6 h                   123                   3.94                     2.67   
6–12 h                  145                   9.78                     9.20   
12–24 h                 776                  15.96                    28.98   

                   mean_absolute_error_h  p90_absolute_error_h  
internal_gap_band                                               
overlap                             6.99                 15.14  
0–1 h                               4.21                  8.49  
1–3 h                               3.57                  6.45  
3–6 h                               5.91  

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

tracks = pd.read_csv(root / "data/processed/engineered_tracks.csv")

tracks["start_time"] = pd.to_datetime(
    tracks["start_time"], utc=True, errors="coerce"
)
tracks["stop_time"] = pd.to_datetime(
    tracks["stop_time"], utc=True, errors="coerce"
)
tracks["distance_km"] = pd.to_numeric(
    tracks["distance_km"], errors="coerce"
)
tracks["track_duration_hours"] = (
    tracks["stop_time"] - tracks["start_time"]
).dt.total_seconds() / 3600

tracks["track_speed_kmh"] = (
    tracks["distance_km"]
    / tracks["track_duration_hours"].where(
        tracks["track_duration_hours"] > 0
    )
)

tracks["flag_missing_id_or_time"] = (
    tracks[["vehicle_id", "tour_id"]].isna().any(axis=1)
    | tracks["start_time"].isna()
    | tracks["stop_time"].isna()
)
tracks["flag_invalid_duration"] = (
    tracks["track_duration_hours"].isna()
    | tracks["track_duration_hours"].le(0)
)
tracks["flag_track_over_24h"] = tracks["track_duration_hours"].gt(24)
tracks["flag_missing_distance"] = tracks["distance_km"].isna()
tracks["flag_negative_distance"] = tracks["distance_km"].lt(0)
tracks["flag_speed_over_120"] = tracks["track_speed_kmh"].gt(120)

flag_columns = [
    "flag_missing_id_or_time",
    "flag_invalid_duration",
    "flag_track_over_24h",
    "flag_missing_distance",
    "flag_negative_distance",
    "flag_speed_over_120",
]
tracks["track_telemetry_flagged"] = tracks[flag_columns].any(axis=1)

keys = ["vehicle_id", "tour_id"]
assignable = tracks.loc[~tracks["flag_missing_id_or_time"]].copy()
assignable = assignable.sort_values(keys + ["start_time"])
assignable["valid_track_hours"] = assignable[
    "track_duration_hours"
].where(assignable["track_duration_hours"] > 0)

GAP_HOURS = 12

assignable["gap_from_previous_h"] = (
    assignable["start_time"]
    - assignable.groupby(keys)["stop_time"].shift()
).dt.total_seconds() / 3600

assignable["new_episode"] = (
    assignable["gap_from_previous_h"] > GAP_HOURS
)
assignable["episode_no"] = (
    assignable.groupby(keys)["new_episode"].cumsum().astype(int)
)
assignable["internal_gap_h"] = assignable[
    "gap_from_previous_h"
].where(~assignable["new_episode"])
assignable["boundary_gap_before_h"] = assignable[
    "gap_from_previous_h"
].where(assignable["new_episode"])

episodes_12h = (
    assignable.groupby(keys + ["episode_no"], as_index=False)
    .agg(
        start_time=("start_time", "min"),
        end_time=("stop_time", "max"),
        total_distance_km=("distance_km", lambda s: s.sum(min_count=1)),
        recorded_track_hours=(
            "valid_track_hours",
            lambda s: s.sum(min_count=1),
        ),
        track_count=("track_id", "size"),
        max_internal_gap_h=("internal_gap_h", "max"),
        boundary_gap_before_h=("boundary_gap_before_h", "first"),
        flagged_track_count=("track_telemetry_flagged", "sum"),
    )
)

episodes_12h["elapsed_span_hours"] = (
    episodes_12h["end_time"] - episodes_12h["start_time"]
).dt.total_seconds() / 3600
episodes_12h["telemetry_flagged"] = (
    episodes_12h["flagged_track_count"] > 0
)

output_path = (
    root / "data/processed/tour_level/telemetry_flagged_episodes_12h.csv"
)
episodes_12h.to_csv(output_path, index=False)

print("Episodes:", len(episodes_12h))
print("Flagged episodes:", int(episodes_12h["telemetry_flagged"].sum()))
print("Saved:", output_path)

Episodes: 18171
Flagged episodes: 280
Saved: /Users/divyshah/Desktop/dt-cargo-vehicle-telemetry-/data/processed/tour_level/telemetry_flagged_episodes_12h.csv


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = episodes_12h.replace([np.inf, -np.inf], np.nan)
df = df.dropna(
    subset=[
        "start_time", "vehicle_id", "total_distance_km",
        "elapsed_span_hours",
    ]
).sort_values("start_time").reset_index(drop=True)

cut = int(0.80 * len(df))
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

# Exclude flagged rows from training; retain all scorable rows in the main holdout.
train = train.loc[~train["telemetry_flagged"]].copy()
inputs = ["vehicle_id", "total_distance_km"]

preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

model = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5,
        random_state=42,
    ),
)

model.fit(train[inputs], train["elapsed_span_hours"])

def report_score(label, scored):
    prediction = model.predict(scored[inputs])
    actual = scored["elapsed_span_hours"]

    print(
        label,
        "n =", len(scored),
        "R² =", round(r2_score(actual, prediction), 3),
        "MAE hours =", round(mean_absolute_error(actual, prediction), 2),
        "RMSE hours =", round(mean_squared_error(actual, prediction) ** 0.5, 2),
    )

scorable_test = test.dropna(subset=inputs + ["elapsed_span_hours"])
clean_test = scorable_test.loc[~scorable_test["telemetry_flagged"]]

print("Training episodes:", len(train))
report_score("All held-out episodes", scorable_test)
report_score("Unflagged held-out episodes (diagnostic)", clean_test)

Training episodes: 14365
All held-out episodes n = 3634 R² = 0.579 MAE hours = 4.47 RMSE hours = 7.77
Unflagged held-out episodes (diagnostic) n = 3526 R² = 0.578 MAE hours = 4.39 RMSE hours = 7.51


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

episodes = pd.read_csv(
    root / "data/processed/tour_level/telemetry_flagged_episodes_12h.csv"
)
episodes["start_time"] = pd.to_datetime(
    episodes["start_time"], utc=True, errors="coerce"
)
episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=[
        "start_time", "vehicle_id",
        "total_distance_km", "elapsed_span_hours",
    ]
).sort_values("start_time").reset_index(drop=True)

cut_train = int(0.60 * len(episodes))
cut_validation = int(0.80 * len(episodes))

train = episodes.iloc[:cut_train].copy()
validation = episodes.iloc[cut_train:cut_validation].copy()
later_holdout = episodes.iloc[cut_validation:].copy()

# Exclude flagged episodes from training only.
train = train.loc[~train["telemetry_flagged"]].copy()

FEATURES = ["vehicle_id", "total_distance_km"]
TARGET = "elapsed_span_hours"

print("Training:", len(train))
print("Validation:", len(validation))
print("Later holdout, reserved:", len(later_holdout))

Training: 10791
Validation: 3634
Later holdout, reserved: 3634


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def regression_scores(actual, predicted):
    return {
        "R2": round(r2_score(actual, predicted), 3),
        "MAE_hours": round(mean_absolute_error(actual, predicted), 2),
        "RMSE_hours": round(mean_squared_error(actual, predicted) ** 0.5, 2),
    }

hgb_preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

hgb_model = make_pipeline(
    hgb_preprocessor,
    HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5,
        random_state=42,
    ),
)

hgb_model.fit(train[FEATURES], train[TARGET])
hgb_predictions = hgb_model.predict(validation[FEATURES])

print("HGB validation:", regression_scores(
    validation[TARGET], hgb_predictions
))

HGB validation: {'R2': 0.335, 'MAE_hours': 5.43, 'RMSE_hours': 11.3}


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import OneHotEncoder, StandardScaler

svm_preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(handle_unknown="ignore"), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

X_train = svm_preprocessor.fit_transform(train[FEATURES])
X_validation = svm_preprocessor.transform(validation[FEATURES])

y_train = train[TARGET].to_numpy()
y_validation = validation[TARGET].to_numpy()

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)

Training matrix: (10791, 67)
Validation matrix: (3634, 67)


In [12]:
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR

kernel_results = []
n_components = min(300, len(train))

for gamma in [0.001, 0.01, 0.1]:
    rbf_map = Nystroem(
        kernel="rbf",
        gamma=gamma,
        n_components=n_components,
        random_state=42,
    )

    Z_train = rbf_map.fit_transform(X_train)
    Z_validation = rbf_map.transform(X_validation)

    for C in [0.1, 1, 10, 100]:
        for epsilon in [0.1, 1, 3]:
            model = LinearSVR(
                C=C,
                epsilon=epsilon,
                max_iter=20000,
                random_state=42,
                dual="auto",
            )
            model.fit(Z_train, y_train)
            predictions = model.predict(Z_validation)

            kernel_results.append({
                "model": "Nystroem + LinearSVR",
                "gamma": gamma,
                "C_or_alpha": C,
                "epsilon": epsilon,
                **regression_scores(y_validation, predictions),
            })

    for alpha in [0.1, 1, 10, 100]:
        model = Ridge(alpha=alpha)
        model.fit(Z_train, y_train)
        predictions = model.predict(Z_validation)

        kernel_results.append({
            "model": "Nystroem + Ridge",
            "gamma": gamma,
            "C_or_alpha": alpha,
            "epsilon": None,
            **regression_scores(y_validation, predictions),
        })

results = pd.DataFrame(kernel_results).sort_values(
    ["R2", "MAE_hours"],
    ascending=[False, True],
)

display(results.head(10))

,model,gamma,C_or_alpha,epsilon,R2,MAE_hours,RMSE_hours
44,Nystroem + Ridge,0.100,0.1,NaN,0.306,5.46,11.54
43,Nystroem + LinearSVR,0.100,100.0,3.0,0.299,4.65,11.60
29,Nystroem + Ridge,0.010,1.0,NaN,0.292,5.96,11.66
12,Nystroem + Ridge,0.001,0.1,NaN,0.289,5.97,11.68
45,Nystroem + Ridge,0.100,1.0,NaN,0.288,5.65,11.69
27,Nystroem + LinearSVR,0.010,100.0,3.0,0.284,4.67,11.72
40,Nystroem + LinearSVR,0.100,10.0,3.0,0.284,4.69,11.72
42,Nystroem + LinearSVR,0.100,100.0,1.0,0.281,4.03,11.75
46,Nystroem + Ridge,0.100,10.0,NaN,0.280,6.06,11.75
24,Nystroem + LinearSVR,0.010,10.0,3.0,0.274,4.71,11.80


In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

episodes = pd.read_csv(
    root / "data/processed/tour_level/telemetry_flagged_episodes_12h.csv"
)
episodes["start_time"] = pd.to_datetime(
    episodes["start_time"], utc=True, errors="coerce"
)
episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=[
        "start_time", "vehicle_id",
        "total_distance_km", "elapsed_span_hours",
    ]
).sort_values("start_time").reset_index(drop=True)

episodes["log_distance_km"] = np.log1p(
    episodes["total_distance_km"].clip(lower=0)
)

cut_train = int(0.60 * len(episodes))
cut_validation = int(0.80 * len(episodes))

train = episodes.iloc[:cut_train].copy()
validation = episodes.iloc[cut_train:cut_validation].copy()
later_holdout = episodes.iloc[cut_validation:].copy()

train = train.loc[~train["telemetry_flagged"]].copy()

print("Training:", len(train))
print("Validation:", len(validation))
print("Later holdout, reserved:", len(later_holdout))

Training: 10791
Validation: 3634
Later holdout, reserved: 3634


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def get_scores(actual, predicted):
    return {
        "R2": round(r2_score(actual, predicted), 3),
        "MAE_hours": round(mean_absolute_error(actual, predicted), 2),
        "RMSE_hours": round(mean_squared_error(actual, predicted) ** 0.5, 2),
    }

feature_sets = {
    "distance": ["vehicle_id", "total_distance_km"],
    "distance_plus_log": [
        "vehicle_id", "total_distance_km", "log_distance_km"
    ],
}

model_configs = [
    (
        "Ridge",
        Ridge(),
        {"model__alpha": np.logspace(-2, 3, 10)},
    ),
    (
        "Lasso",
        Lasso(max_iter=50000, tol=1e-4),
        {"model__alpha": np.logspace(-4, 1, 10)},
    ),
    (
        "ElasticNet",
        ElasticNet(max_iter=50000, tol=1e-4),
        {
            "model__alpha": np.logspace(-4, 1, 10),
            "model__l1_ratio": [0.1, 0.5, 0.9],
        },
    ),
]

results = []

for feature_name, feature_columns in feature_sets.items():
    numeric_columns = [
        col for col in feature_columns if col != "vehicle_id"
    ]

    for model_name, estimator, param_grid in model_configs:
        preprocessor = ColumnTransformer([
            ("vehicle", OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False,
            ), ["vehicle_id"]),
            ("numeric", StandardScaler(), numeric_columns),
        ])

        pipeline = Pipeline([
            ("prep", preprocessor),
            ("model", estimator),
        ])

        search = GridSearchCV(
            pipeline,
            param_grid=param_grid,
            scoring="r2",
            cv=TimeSeriesSplit(n_splits=4),
            n_jobs=1,
        )
        search.fit(train[feature_columns], train["elapsed_span_hours"])

        prediction = search.predict(validation[feature_columns])

        results.append({
            "model": model_name,
            "features": feature_name,
            "CV_R2": round(search.best_score_, 3),
            **get_scores(validation["elapsed_span_hours"], prediction),
            "best_parameters": search.best_params_,
        })

# HGB benchmark, evaluated on the same validation rows.
hgb_preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance", StandardScaler(), ["total_distance_km"]),
])

hgb_model = make_pipeline(
    hgb_preprocessor,
    HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5,
        random_state=42,
    ),
)

hgb_features = ["vehicle_id", "total_distance_km"]
hgb_model.fit(train[hgb_features], train["elapsed_span_hours"])
hgb_prediction = hgb_model.predict(validation[hgb_features])

results.append({
    "model": "HGB benchmark",
    "features": "distance",
    "CV_R2": np.nan,
    **get_scores(validation["elapsed_span_hours"], hgb_prediction),
    "best_parameters": "fixed settings",
})

/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These un

In [15]:
results_df = pd.DataFrame(results).sort_values(
    ["R2", "MAE_hours"],
    ascending=[False, True],
)

display(results_df)

,model,features,CV_R2,R2,MAE_hours,RMSE_hours,best_parameters
6,HGB benchmark,distance,NaN,0.335,5.43,11.30,fixed settings
5,ElasticNet,distance_plus_log,0.169,0.254,5.86,11.96,"{'model__alpha': 0.001291549665014884, 'model_..."
2,ElasticNet,distance,0.166,0.252,5.84,11.98,"{'model__alpha': 0.001291549665014884, 'model_..."
3,Ridge,distance_plus_log,0.169,0.249,5.81,12.01,{'model__alpha': 5.994842503189409}
0,Ridge,distance,0.167,0.247,5.80,12.03,{'model__alpha': 5.994842503189409}
4,Lasso,distance_plus_log,0.168,0.244,5.82,12.04,{'model__alpha': 0.004641588833612782}
1,Lasso,distance,0.166,0.243,5.81,12.06,{'model__alpha': 0.004641588833612782}


In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

episodes = pd.read_csv(
    root / "data/processed/tour_level/telemetry_flagged_episodes_12h.csv"
)
episodes["start_time"] = pd.to_datetime(
    episodes["start_time"], utc=True, errors="coerce"
)
episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=[
        "start_time", "vehicle_id",
        "total_distance_km", "elapsed_span_hours",
    ]
).sort_values("start_time").reset_index(drop=True)

cut = int(0.80 * len(episodes))
train = episodes.iloc[:cut].copy()
test = episodes.iloc[cut:].copy()

# Exclude flagged episodes from training; score every scorable test episode.
train = train.loc[~train["telemetry_flagged"]].copy()

print("Training:", len(train))
print("Holdout:", len(test))

Training: 14365
Holdout: 3634


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

results = []

for degree in [1, 2, 3, 4]:
    preprocessor = ColumnTransformer([
        ("vehicle", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False,
        ), ["vehicle_id"]),
        ("distance_poly", Pipeline([
            ("scale", StandardScaler()),
            ("poly", PolynomialFeatures(
                degree=degree,
                include_bias=False,
            )),
        ]), ["total_distance_km"]),
    ])

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", Ridge()),
    ])

    search = GridSearchCV(
        pipeline,
        param_grid={"model__alpha": np.logspace(-2, 3, 10)},
        scoring="r2",
        cv=TimeSeriesSplit(n_splits=4),
        n_jobs=1,
    )

    inputs = ["vehicle_id", "total_distance_km"]
    search.fit(train[inputs], train["elapsed_span_hours"])
    predictions = search.predict(test[inputs])

    results.append({
        "degree": degree,
        "CV_R2": round(search.best_score_, 3),
        "holdout_R2": round(
            r2_score(test["elapsed_span_hours"], predictions), 3
        ),
        "MAE_hours": round(
            mean_absolute_error(test["elapsed_span_hours"], predictions), 2
        ),
        "RMSE_hours": round(
            mean_squared_error(
                test["elapsed_span_hours"], predictions
            ) ** 0.5,
            2,
        ),
        "best_alpha": search.best_params_["model__alpha"],
    })

display(pd.DataFrame(results))

/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These un

,degree,CV_R2,holdout_R2,MAE_hours,RMSE_hours,best_alpha
0,1,0.188,0.456,7.16,8.84,21.544347
1,2,0.209,0.496,6.55,8.50,21.544347
2,3,0.204,0.091,10.23,11.42,1.668101
3,4,0.190,-0.441,13.30,14.38,0.464159


In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

episodes = pd.read_csv(
    root / "data/processed/tour_level/telemetry_flagged_episodes_12h.csv"
)
episodes["start_time"] = pd.to_datetime(
    episodes["start_time"], utc=True, errors="coerce"
)
episodes = episodes.replace([np.inf, -np.inf], np.nan)
episodes = episodes.dropna(
    subset=[
        "start_time", "vehicle_id",
        "total_distance_km", "elapsed_span_hours",
    ]
).sort_values("start_time").reset_index(drop=True)

cut = int(0.80 * len(episodes))
train = episodes.iloc[:cut].copy()
holdout = episodes.iloc[cut:].copy()

# Exclude flagged episodes from training; keep all scorable holdout episodes.
train = train.loc[~train["telemetry_flagged"]].copy()

features = ["vehicle_id", "total_distance_km"]
target = "elapsed_span_hours"

print("Training:", len(train))
print("Holdout:", len(holdout))

Training: 14365
Holdout: 3634


In [19]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

preprocessor = ColumnTransformer([
    ("vehicle", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False,
    ), ["vehicle_id"]),
    ("distance_poly", Pipeline([
        ("scale_input", StandardScaler()),
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False,
        )),
        ("scale_polynomial_terms", StandardScaler()),
    ]), ["total_distance_km"]),
])

sgd = SGDRegressor(
    loss="squared_error",
    penalty="l2",
    alpha=0.001,
    learning_rate="adaptive",
    eta0=0.001,
    max_iter=5000,
    tol=1e-4,
    random_state=42,
    average=True,
)

model = Pipeline([
    ("prep", preprocessor),
    ("sgd", TransformedTargetRegressor(
        regressor=sgd,
        transformer=StandardScaler(),
    )),
])

model.fit(train[features], train[target])
predictions = model.predict(holdout[features])
actual = holdout[target]

print("R²:", round(r2_score(actual, predictions), 3))
print("MAE hours:", round(mean_absolute_error(actual, predictions), 2))
print("RMSE hours:", round(mean_squared_error(actual, predictions) ** 0.5, 2))

R²: 0.535
MAE hours: 6.04
RMSE hours: 8.17


/Users/divyshah/Downloads/Python DSA/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [20]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

known_vehicles = set(train["vehicle_id"])
unknown_rows = ~holdout["vehicle_id"].isin(known_vehicles)

print("Unseen holdout vehicles:", holdout.loc[unknown_rows, "vehicle_id"].nunique())
print("Holdout episodes with unseen vehicles:", int(unknown_rows.sum()))

features = ["total_distance_km"]

model = Pipeline([
    ("polynomial_distance", Pipeline([
        ("scale_input", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale_terms", StandardScaler()),
    ])),
    ("sgd", TransformedTargetRegressor(
        regressor=SGDRegressor(
            loss="squared_error",
            penalty="l2",
            alpha=0.001,
            learning_rate="adaptive",
            eta0=0.001,
            max_iter=5000,
            tol=1e-4,
            random_state=42,
            average=True,
        ),
        transformer=StandardScaler(),
    )),
])

model.fit(train[features], train["elapsed_span_hours"])
predictions = model.predict(holdout[features])
actual = holdout["elapsed_span_hours"]

print("R²:", round(r2_score(actual, predictions), 3))
print("MAE hours:", round(mean_absolute_error(actual, predictions), 2))
print("RMSE hours:", round(mean_squared_error(actual, predictions) ** 0.5, 2))

Unseen holdout vehicles: 97
Holdout episodes with unseen vehicles: 3367
R²: 0.621
MAE hours: 4.93
RMSE hours: 7.38
